# B6：Agent 可觀測性與線上除錯 — 生產環境架構決策

> **場景：** 金融科技公司的貸款審查 Agent（每日處理 2,000 件申請）。  
> 上線後出現 **15% 神秘失敗率**，但傳統日誌只記了 HTTP 200/500，  
> 完全看不出是 LLM 幻覺、工具呼叫失敗、還是 Context 超限導致的問題。

## 這本 Notebook 要回答的核心問題

每個架構決策都會問：**「為什麼用 X，不用 Y？」**

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 追蹤粒度 | Granular Span（每次 tool call / LLM call 獨立 span） | 只記 request-level log |
| 採樣策略 | Tail-based Sampling（失敗 100%，成功 5%） | 全量採樣 |
| 效能指標 | LLM-native 指標（TTFT、tokens/sec、cost/request） | 只看 HTTP latency |
| 錯誤分類 | 自動 Error Taxonomy（幻覺/工具失敗/context超限） | 人工分析原始 log |
| 追蹤後端 | Cloud Trace + Cloud Logging（GCP native） | Jaeger/Zipkin 自建 |

In [ ]:
import time
import json
import uuid
import random
import hashlib
from typing import Optional, Literal
from dataclasses import dataclass, field
from contextvars import ContextVar
from enum import Enum
from collections import defaultdict, Counter

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

print("\n場景：金融科技貸款審查 Agent")
print("每日申請：2,000 件 | 神秘失敗率：15% | 需要定位根因")

## 系統架構全貌

```
貸款申請進入
      │
      ▼
┌─────────────────────────────────────────────────────┐
│  Loan Review Agent                                  │
│                                                     │
│   LLM Call ──────────┐                             │
│   (信用分析)          │                             │
│                       ▼                             │
│   Tool Call ──► [Span Collector] ──► Tail Sampler  │
│   (查徵信局)          │                   │         │
│                       │         失敗 100% │         │
│   Tool Call           │         成功  5%  │         │
│   (查收入記錄)         │                   │         │
│                       │                   │         │
│   Context Build ──────┘                   │         │
│   (組裝申請資料)                           │         │
└─────────────────────────────────────────┬─┘         │
                                          │            │
              ┌───────────────────────────┘            │
              ▼                                        │
   ┌──────────────────────┐                            │
   │   Observability      │                            │
   │   Backend (GCP)      │                            │
   │                      │                            │
   │  Cloud Trace         │  ← 分散式追蹤 (spans)      │
   │  Cloud Logging       │  ← 結構化日誌              │
   │  Cloud Monitoring    │  ← TTFT / cost / tokens   │
   │  Error Taxonomy      │  ← 自動錯誤分類            │
   └──────────────────────┘
```

## 決策 1：Granular Span vs. Request-level Log

**❌ 被拒絕的方案：Request-level Log**  
傳統 APM 只記「這個請求花了 2.3 秒，最終回傳 HTTP 500」。  
對 Agent 完全沒用——一個 Agent 請求包含 3-5 次 LLM Call + 4-6 次 Tool Call，  
失敗可能在第幾個步驟根本看不出來。

**問題：**
- 知道「失敗了」，但不知道是哪一步失敗（第 2 次 LLM call？第 3 次 tool call？）
- 無法比較「成功的請求」和「失敗的請求」在 token 使用上有何差異
- 無法計算每個工具呼叫的 P95 延遲
- 根本無法 debug 15% 的神秘失敗

**✅ 選擇的方案：每個 LLM Call / Tool Call 獨立 Span**

In [ ]:
@dataclass
class TraceSpan:
    """
    每個 LLM Call 或 Tool Call 的獨立追蹤單元。
    
    關鍵欄位：
    - parent_span_id：建立 call tree，知道哪個 tool call 是哪個 LLM call 觸發的
    - token_count：LLM 的核心成本指標
    - model：同一請求可能混用 Flash / Pro
    - cost_usd：每個 span 的金錢成本
    """
    span_id:       str
    trace_id:      str           # 整個請求的唯一 ID
    parent_span_id: Optional[str]
    operation:     str           # llm_call / tool_call / context_build
    
    # 時間
    start_time:    float = field(default_factory=time.monotonic)
    end_time:      Optional[float] = None
    duration_ms:   Optional[float] = None

    # LLM 特有欄位
    model:         Optional[str] = None
    input_tokens:  Optional[int] = None
    output_tokens: Optional[int] = None
    ttft_ms:       Optional[float] = None   # Time to First Token
    cost_usd:      Optional[float] = None

    # Tool 特有欄位
    tool_name:     Optional[str] = None
    tool_success:  Optional[bool] = None
    tool_error:    Optional[str] = None

    # Context 特有欄位
    context_tokens: Optional[int] = None
    context_limit:  Optional[int] = None

    # 通用
    status:        str = "ok"   # ok / error / timeout
    attributes:    dict = field(default_factory=dict)

    def finish(self, status: str = "ok"):
        self.end_time   = time.monotonic()
        self.duration_ms = (self.end_time - self.start_time) * 1000
        self.status     = status

    def to_dict(self) -> dict:
        return {
            k: v for k, v in self.__dict__.items() if v is not None
        }


# ContextVar：讓 span propagation 在 async 環境下安全
# （asyncio 每個 task 有獨立的 ContextVar 值，不會互相污染）
current_trace_id:    ContextVar[Optional[str]] = ContextVar("current_trace_id",    default=None)
current_span_id:     ContextVar[Optional[str]] = ContextVar("current_span_id",     default=None)


class SpanCollector:
    """收集所有 span，準備交給 Tail Sampler 決定是否要保留。"""
    
    def __init__(self):
        self._spans: dict[str, list[TraceSpan]] = defaultdict(list)  # trace_id → spans

    def record(self, span: TraceSpan):
        self._spans[span.trace_id].append(span)

    def get_trace(self, trace_id: str) -> list[TraceSpan]:
        return self._spans.get(trace_id, [])

    def all_traces(self) -> dict[str, list[TraceSpan]]:
        return dict(self._spans)


collector = SpanCollector()


def instrument_llm_call(trace_id: str, parent_span_id: str,
                         model: str, input_tokens: int, output_tokens: int,
                         ttft_ms: float, status: str = "ok") -> TraceSpan:
    """為一次 LLM Call 建立並記錄 span。"""
    # 模型定價（每 1K tokens，USD）
    PRICING = {
        "gemini-1.5-flash": {"input": 0.000075, "output": 0.0003},
        "gemini-1.5-pro":   {"input": 0.00125,  "output": 0.005},
    }
    p = PRICING.get(model, {"input": 0.001, "output": 0.003})
    cost = (input_tokens * p["input"] + output_tokens * p["output"]) / 1000

    span = TraceSpan(
        span_id        = str(uuid.uuid4())[:8],
        trace_id       = trace_id,
        parent_span_id = parent_span_id,
        operation      = "llm_call",
        model          = model,
        input_tokens   = input_tokens,
        output_tokens  = output_tokens,
        ttft_ms        = ttft_ms,
        cost_usd       = round(cost, 6),
    )
    span.finish(status)
    collector.record(span)
    return span


def instrument_tool_call(trace_id: str, parent_span_id: str,
                          tool_name: str, duration_ms: float,
                          success: bool, error: Optional[str] = None) -> TraceSpan:
    """為一次 Tool Call 建立並記錄 span。"""
    span = TraceSpan(
        span_id        = str(uuid.uuid4())[:8],
        trace_id       = trace_id,
        parent_span_id = parent_span_id,
        operation      = "tool_call",
        tool_name      = tool_name,
        tool_success   = success,
        tool_error     = error,
    )
    span.duration_ms = duration_ms
    span.finish("ok" if success else "error")
    collector.record(span)
    return span


# ── 模擬一個完整的貸款審查 Agent 追蹤 ────────────────────────────────────
def simulate_loan_review_trace(application_id: str,
                                scenario: str = "normal") -> str:
    """
    模擬貸款審查 Agent 的完整執行追蹤。
    scenario: normal / tool_failure / context_overflow / hallucination / timeout
    """
    trace_id   = hashlib.md5(application_id.encode()).hexdigest()[:12]
    root_span  = str(uuid.uuid4())[:8]  # Agent 根 span

    # Step 1：建構 Context（組裝申請資料）
    ctx_tokens  = 128000 if scenario == "context_overflow" else random.randint(2000, 6000)
    ctx_span = TraceSpan(
        span_id        = str(uuid.uuid4())[:8],
        trace_id       = trace_id,
        parent_span_id = root_span,
        operation      = "context_build",
        context_tokens = ctx_tokens,
        context_limit  = 32768,
    )
    ctx_span.duration_ms = random.uniform(5, 20)
    ctx_span.finish("error" if scenario == "context_overflow" else "ok")
    collector.record(ctx_span)

    if scenario == "context_overflow":
        return trace_id   # 直接失敗，後續步驟沒有執行

    # Step 2：Tool Call — 查詢聯徵中心
    tool_ok   = (scenario != "tool_failure")
    tool_err  = "JCIC API timeout after 5000ms" if scenario == "tool_failure" else None
    instrument_tool_call(trace_id, root_span, "query_jcic",
                         duration_ms = 5200 if scenario == "tool_failure" else random.uniform(200, 800),
                         success     = tool_ok,
                         error       = tool_err)

    if scenario == "tool_failure":
        return trace_id

    # Step 3：Tool Call — 查詢收入記錄
    instrument_tool_call(trace_id, root_span, "query_income_records",
                         duration_ms = random.uniform(100, 400),
                         success     = True)

    # Step 4：LLM Call — 信用分析
    ttft = random.uniform(800, 1200) if scenario == "timeout" else random.uniform(120, 350)
    llm_status = "timeout" if scenario == "timeout" else "ok"
    instrument_llm_call(
        trace_id, root_span,
        model        = "gemini-1.5-pro",
        input_tokens = random.randint(3000, 5000),
        output_tokens = 0 if scenario == "timeout" else random.randint(200, 500),
        ttft_ms      = ttft,
        status       = llm_status,
    )

    if scenario == "timeout":
        return trace_id

    # Step 5：LLM Call — 審查決策（可能幻覺）
    if scenario == "hallucination":
        # 幻覺的特徵：output tokens 過少（截斷），或過多（胡亂輸出）
        out_tokens = random.choice([12, 15, 8])  # 結構化輸出嚴重不足
    else:
        out_tokens = random.randint(150, 400)

    instrument_llm_call(
        trace_id, root_span,
        model        = "gemini-1.5-pro",
        input_tokens = random.randint(4000, 6000),
        output_tokens = out_tokens,
        ttft_ms      = random.uniform(150, 300),
        status       = "ok",
    )

    return trace_id


# ── 生成模擬資料：100 筆貸款審查 ─────────────────────────────────────────
scenarios = (["normal"] * 85 +
             ["tool_failure"] * 6 +
             ["context_overflow"] * 4 +
             ["hallucination"] * 3 +
             ["timeout"] * 2)
random.shuffle(scenarios)

trace_ids_by_scenario: dict[str, list[str]] = defaultdict(list)
for i, sc in enumerate(scenarios):
    tid = simulate_loan_review_trace(f"LOAN-{i:04d}", sc)
    trace_ids_by_scenario[sc].append(tid)

all_traces = collector.all_traces()
print(f"已模擬 {len(all_traces)} 筆追蹤，共 {sum(len(v) for v in all_traces.values())} 個 span")
print()

# 展示一筆正常追蹤的 span 結構
sample_tid = trace_ids_by_scenario["normal"][0]
print(f"正常審查追蹤 [{sample_tid}] 的 span 結構：")
print("─" * 60)
for sp in collector.get_trace(sample_tid):
    indent = "  " if sp.parent_span_id else ""
    if sp.operation == "llm_call":
        print(f"{indent}[LLM] {sp.model} | in={sp.input_tokens}tok out={sp.output_tokens}tok "
              f"ttft={sp.ttft_ms:.0f}ms cost=${sp.cost_usd:.5f}")
    elif sp.operation == "tool_call":
        status_icon = "✅" if sp.tool_success else "❌"
        print(f"{indent}[Tool] {status_icon} {sp.tool_name} | {sp.duration_ms:.0f}ms")
    elif sp.operation == "context_build":
        util = sp.context_tokens / sp.context_limit * 100
        print(f"{indent}[Ctx]  context={sp.context_tokens:,}tok / {sp.context_limit:,} ({util:.0f}% used)")

# 展示一筆失敗追蹤
fail_tid = trace_ids_by_scenario["tool_failure"][0]
print(f"\n工具失敗追蹤 [{fail_tid}] 的 span 結構：")
print("─" * 60)
for sp in collector.get_trace(fail_tid):
    if sp.operation == "tool_call":
        print(f"  [Tool] ❌ {sp.tool_name} | {sp.duration_ms:.0f}ms | error: {sp.tool_error}")
    elif sp.operation == "context_build":
        print(f"  [Ctx]  context={sp.context_tokens:,}tok")

## 決策 2：Tail-based Sampling vs. 全量採樣

**❌ 被拒絕的方案：全量採樣**  
每日 2,000 筆申請 × 每筆 5-8 個 span = **10,000–16,000 個 span/日**。  
Cloud Trace 定價：$0.20 per 1M spans；聽起來便宜，但結構化日誌另外計費，  
而且 99% 的正常請求根本沒有 debug 價值，全存只是浪費存儲配額。

**❌ Head-based Sampling 也不夠好**  
在請求進入時就決定要不要採樣（例如隨機 10%）。  
問題：你在決定的那一刻不知道這個請求最終會成功還是失敗。  
結果：有 90% 機率丟掉了你最需要調試的失敗案例。

**✅ 選擇的方案：Tail-based Sampling（結果知道了再決定要不要留）**
- 失敗 / 慢請求（P95 以上）：100% 採樣
- 正常成功請求：5% 採樣（保留基線，用於 drift 偵測）

In [ ]:
class TailBasedSampler:
    """
    Tail-based Sampling：等請求完成後再決定是否保留整條 trace。
    
    原理：
    1. 所有 span 先暫存在記憶體（最多 60 秒 buffer）
    2. 當 root span 完成（或 timeout），評估整條 trace 的特徵
    3. 根據策略決定：export（送到 Cloud Trace）或 drop（丟棄）
    
    Tail-based 的代價：需要暫存 in-flight spans，有記憶體壓力。
    對策：設定 max_buffer_size，超過則強制 export（寧可多存，不可漏失敗）。
    """

    def __init__(
        self,
        error_sample_rate:   float = 1.00,  # 失敗：100% 保留
        slow_sample_rate:    float = 1.00,  # 慢請求：100% 保留
        success_sample_rate: float = 0.05,  # 正常成功：5% 保留
        slow_threshold_ms:   float = 8000,  # 超過 8 秒視為慢請求
    ):
        self.error_sample_rate   = error_sample_rate
        self.slow_sample_rate    = slow_sample_rate
        self.success_sample_rate = success_sample_rate
        self.slow_threshold_ms   = slow_threshold_ms

        self._exported_count = 0
        self._dropped_count  = 0
        self._exported_traces: list[str] = []   # 模擬 export 目的地

    def _classify_trace(self, spans: list[TraceSpan]) -> dict:
        """分析一條 trace 的特徵，決定分類。"""
        has_error   = any(sp.status in ("error", "timeout") for sp in spans)
        total_ms    = sum(sp.duration_ms for sp in spans if sp.duration_ms)
        is_slow     = total_ms > self.slow_threshold_ms
        total_cost  = sum(sp.cost_usd for sp in spans if sp.cost_usd)
        total_tokens = sum(
            (sp.input_tokens or 0) + (sp.output_tokens or 0)
            for sp in spans
        )
        return {
            "has_error":    has_error,
            "is_slow":      is_slow,
            "total_ms":     total_ms,
            "total_cost":   total_cost,
            "total_tokens": total_tokens,
            "span_count":   len(spans),
        }

    def should_export(self, trace_id: str, spans: list[TraceSpan]) -> tuple[bool, str]:
        """
        決定是否 export 這條 trace。
        回傳 (要不要保留, 原因)。
        """
        if not spans:
            return False, "empty_trace"

        info = self._classify_trace(spans)

        # 錯誤 trace：永遠保留
        if info["has_error"]:
            if random.random() < self.error_sample_rate:
                return True, "error_always_sample"

        # 慢請求：永遠保留
        if info["is_slow"]:
            if random.random() < self.slow_sample_rate:
                return True, "slow_request_always_sample"

        # 正常成功：5% 採樣
        if random.random() < self.success_sample_rate:
            return True, "success_baseline_sample"

        return False, "dropped_normal"

    def process_all(self, all_traces: dict[str, list[TraceSpan]]):
        """批次處理所有 trace（模擬真實的 buffer 處理）。"""
        reason_counts: Counter = Counter()

        for trace_id, spans in all_traces.items():
            keep, reason = self.should_export(trace_id, spans)
            reason_counts[reason] += 1
            if keep:
                self._exported_count += 1
                self._exported_traces.append(trace_id)
            else:
                self._dropped_count += 1

        return reason_counts


sampler = TailBasedSampler()
reason_counts = sampler.process_all(all_traces)

total = sampler._exported_count + sampler._dropped_count
export_rate = sampler._exported_count / total * 100

print("Tail-based Sampling 結果：")
print("=" * 55)
print(f"  總追蹤數：        {total}")
print(f"  已 Export：       {sampler._exported_count} ({export_rate:.1f}%)")
print(f"  已丟棄：          {sampler._dropped_count} ({100 - export_rate:.1f}%)")
print()
print("Export 原因分布：")
for reason, count in reason_counts.most_common():
    print(f"  {reason:<35} {count:>3} 筆")

# 成本對比
full_span_count  = sum(len(v) for v in all_traces.values())
exported_span_count = sum(
    len(all_traces[tid])
    for tid in sampler._exported_traces
    if tid in all_traces
)

print()
print("儲存成本分析（每日 2,000 筆申請，比例換算）：")
scale = 2000 / 100   # 模擬比例換算
daily_full   = full_span_count  * scale
daily_export = exported_span_count * scale
# Cloud Trace: $0.20/1M spans; Cloud Logging: ~$0.50/GB
cost_full   = daily_full   / 1_000_000 * 0.20 * 30  # 月費
cost_export = daily_export / 1_000_000 * 0.20 * 30
print(f"  全量採樣月 span 數：  {daily_full * 30:,.0f}  → 月費 ~${cost_full:.2f}")
print(f"  Tail 採樣月 span 數： {daily_export * 30:,.0f}  → 月費 ~${cost_export:.2f}")
print(f"  節省：              ~{(1 - cost_export / cost_full) * 100:.0f}% 存儲費用")
print(f"  失敗追蹤保留率：    100%（全部失敗案例都能 debug）")

## 決策 3：LLM-native Metrics vs. 傳統 HTTP Metrics

**❌ 被拒絕的方案：只看 HTTP Latency**  
傳統 APM（Datadog、New Relic）告訴你：「P95 latency = 4.2s」。  
問題是這完全無法指導優化方向：
- 是 LLM TTFT 太慢？還是 DB tool call 慢？
- 是 output tokens 過多？還是模型推理慢？
- 哪個申請的 cost 最高？

**✅ 選擇的方案：LLM-native Metrics**
- **TTFT**（Time to First Token）：感知延遲的核心，比 total latency 更能反映用戶體驗
- **tokens/sec**：模型推理吞吐量，用於偵測模型端瓶頸
- **cost/request**：每筆申請的實際金錢成本，用於 ROI 分析
- **tool_call_count**：LLM 呼叫了幾次工具，間接反映任務複雜度

In [ ]:
class MetricsCollector:
    """
    從 span 資料提取 LLM-native 效能指標。
    
    這些指標回答 HTTP metrics 回答不了的問題：
    - 我的模型選型對不對？（tokens/sec 比較 Flash vs Pro）
    - 哪個工具是瓶頸？（tool P95 latency 分布）
    - 每筆審查花多少錢？（cost/request 趨勢）
    """

    def __init__(self, all_traces: dict[str, list[TraceSpan]]):
        self.traces = all_traces

    def _percentile(self, values: list[float], p: float) -> float:
        if not values:
            return 0.0
        values_sorted = sorted(values)
        idx = int(len(values_sorted) * p / 100)
        return values_sorted[min(idx, len(values_sorted) - 1)]

    def llm_latency_report(self) -> dict:
        """LLM 呼叫的延遲分布。"""
        ttfts_by_model: dict[str, list[float]] = defaultdict(list)
        tps_by_model:   dict[str, list[float]] = defaultdict(list)  # tokens/sec

        for spans in self.traces.values():
            for sp in spans:
                if sp.operation != "llm_call" or not sp.model:
                    continue
                if sp.ttft_ms:
                    ttfts_by_model[sp.model].append(sp.ttft_ms)
                if sp.output_tokens and sp.duration_ms and sp.duration_ms > 0:
                    tps = sp.output_tokens / (sp.duration_ms / 1000)
                    tps_by_model[sp.model].append(tps)

        report = {}
        for model in ttfts_by_model:
            ttfts = ttfts_by_model[model]
            tps   = tps_by_model.get(model, [])
            report[model] = {
                "ttft_p50_ms":   round(self._percentile(ttfts, 50), 1),
                "ttft_p95_ms":   round(self._percentile(ttfts, 95), 1),
                "ttft_p99_ms":   round(self._percentile(ttfts, 99), 1),
                "tokens_per_sec_p50": round(self._percentile(tps, 50), 1) if tps else "N/A",
                "sample_count":  len(ttfts),
            }
        return report

    def tool_latency_report(self) -> dict:
        """各工具呼叫的延遲和成功率。"""
        durations: dict[str, list[float]] = defaultdict(list)
        errors:    dict[str, int]         = defaultdict(int)

        for spans in self.traces.values():
            for sp in spans:
                if sp.operation != "tool_call" or not sp.tool_name:
                    continue
                if sp.duration_ms:
                    durations[sp.tool_name].append(sp.duration_ms)
                if not sp.tool_success:
                    errors[sp.tool_name] += 1

        report = {}
        for tool, durs in durations.items():
            report[tool] = {
                "p50_ms":    round(self._percentile(durs, 50), 1),
                "p95_ms":    round(self._percentile(durs, 95), 1),
                "error_count": errors.get(tool, 0),
                "error_rate":  f"{errors.get(tool, 0) / len(durs) * 100:.1f}%",
                "call_count":  len(durs),
            }
        return report

    def cost_report(self) -> dict:
        """每筆審查的成本分析。"""
        costs = []
        for spans in self.traces.values():
            trace_cost = sum(sp.cost_usd for sp in spans if sp.cost_usd)
            if trace_cost > 0:
                costs.append(trace_cost)

        if not costs:
            return {}
        return {
            "avg_cost_usd":        round(sum(costs) / len(costs), 5),
            "p95_cost_usd":        round(self._percentile(costs, 95), 5),
            "max_cost_usd":        round(max(costs), 5),
            "daily_cost_2000_usd": round(sum(costs) / len(costs) * 2000, 2),
            "monthly_cost_usd":    round(sum(costs) / len(costs) * 2000 * 30, 2),
        }

    def context_utilization_report(self) -> dict:
        """Context 視窗使用率分析。"""
        utilizations = []
        overflow_count = 0
        for spans in self.traces.values():
            for sp in spans:
                if sp.operation == "context_build" and sp.context_tokens and sp.context_limit:
                    util = sp.context_tokens / sp.context_limit
                    utilizations.append(util)
                    if sp.context_tokens > sp.context_limit:
                        overflow_count += 1
        return {
            "avg_utilization":    f"{self._percentile(utilizations, 50) * 100:.1f}%",
            "p95_utilization":    f"{self._percentile(utilizations, 95) * 100:.1f}%",
            "overflow_count":     overflow_count,
            "overflow_rate":      f"{overflow_count / len(utilizations) * 100:.1f}%" if utilizations else "0%",
        }


metrics = MetricsCollector(all_traces)

print("LLM 延遲指標（HTTP metrics 看不到的）：")
print("─" * 60)
for model, stats in metrics.llm_latency_report().items():
    print(f"  {model}")
    print(f"    TTFT  P50={stats['ttft_p50_ms']}ms  P95={stats['ttft_p95_ms']}ms  P99={stats['ttft_p99_ms']}ms")
    print(f"    tokens/sec P50={stats['tokens_per_sec_p50']}  sample={stats['sample_count']}")

print()
print("工具呼叫延遲 + 錯誤率：")
print("─" * 60)
for tool, stats in metrics.tool_latency_report().items():
    print(f"  {tool:<30} P50={stats['p50_ms']:>6}ms  P95={stats['p95_ms']:>7}ms  "
          f"失敗率={stats['error_rate']}  n={stats['call_count']}")

print()
print("成本分析（LLM-native，非 infrastructure cost）：")
print("─" * 60)
for k, v in metrics.cost_report().items():
    label = {
        "avg_cost_usd":        "平均每筆成本",
        "p95_cost_usd":        "P95 成本",
        "max_cost_usd":        "最高單筆",
        "daily_cost_2000_usd": "每日 2,000 筆總成本",
        "monthly_cost_usd":    "月成本估算",
    }.get(k, k)
    print(f"  {label:<25} ${v}")

print()
print("Context 視窗使用率：")
print("─" * 60)
for k, v in metrics.context_utilization_report().items():
    print(f"  {k:<25} {v}")

## 決策 4：自動 Error Taxonomy vs. 人工分析原始 log

**❌ 被拒絕的方案：人工閱讀原始 log**  
15% 的失敗率 × 2,000 筆/日 = **每日 300 筆失敗需要人工分析**。  
傳統方法：工程師打開 log，看一堆 `ERROR: LLM returned unexpected format`，  
無法快速判斷這是幻覺（LLM 問題）還是工具失敗（DB 問題）還是 context 超限（設計問題）。

**✅ 選擇的方案：自動 Error Taxonomy**  
定義 5 種錯誤類型，從 span 資料自動分類，每種錯誤對應不同的修復方向：

| 錯誤類型 | 診斷依據 | 修復方向 |
|----------|----------|----------|
| `HALLUCINATION` | output_tokens 過少（結構化輸出截斷） | 改 Prompt + 加 Output Parser |
| `TOOL_FAILURE` | tool span status=error | 修 Tool 實作 / 加 Retry |
| `CONTEXT_OVERFLOW` | context_tokens > context_limit | 縮 Context / 換大 window 模型 |
| `TIMEOUT` | LLM span status=timeout | 換 Flash / 加 timeout + retry |
| `PARSING_ERROR` | 沒有 output span + LLM status=ok | 修 Output Parser |

In [ ]:
class ErrorType(Enum):
    HALLUCINATION     = "HALLUCINATION"      # LLM 輸出結構不符預期
    TOOL_FAILURE      = "TOOL_FAILURE"        # 工具呼叫失敗（DB、API）
    CONTEXT_OVERFLOW  = "CONTEXT_OVERFLOW"    # Context 超過模型上限
    TIMEOUT           = "TIMEOUT"             # LLM 或工具超時
    PARSING_ERROR     = "PARSING_ERROR"       # 輸出解析失敗
    UNKNOWN           = "UNKNOWN"             # 無法分類


@dataclass
class ErrorDiagnosis:
    trace_id:    str
    error_type:  ErrorType
    confidence:  float          # 0.0 - 1.0
    evidence:    list[str]      # 診斷依據（可讀文字）
    suggestion:  str            # 行動建議


class ErrorClassifier:
    """
    從 span 資料自動分類錯誤類型。
    
    設計原則：
    - 優先順序：CONTEXT_OVERFLOW > TOOL_FAILURE > TIMEOUT > HALLUCINATION > PARSING_ERROR
    - 優先順序基於：越明確的 error signal 越優先（overflow 最確定）
    - confidence 反映診斷的把握程度
    """

    # 幻覺判斷閾值：output tokens 過少視為截斷/幻覺
    HALLUCINATION_OUTPUT_THRESHOLD = 30

    def classify(self, trace_id: str, spans: list[TraceSpan]) -> Optional[ErrorDiagnosis]:
        """分析一條 trace，若有錯誤則回傳診斷結果。"""
        if not spans:
            return None

        # 沒有任何錯誤 span → 正常請求
        has_any_error = any(sp.status in ("error", "timeout") for sp in spans)
        if not has_any_error:
            return None

        # Rule 1：Context Overflow（最明確）
        ctx_spans = [sp for sp in spans if sp.operation == "context_build"]
        for sp in ctx_spans:
            if sp.context_tokens and sp.context_limit:
                if sp.context_tokens > sp.context_limit:
                    return ErrorDiagnosis(
                        trace_id   = trace_id,
                        error_type = ErrorType.CONTEXT_OVERFLOW,
                        confidence = 0.98,
                        evidence   = [
                            f"context_tokens={sp.context_tokens:,} 超過 context_limit={sp.context_limit:,}",
                            f"overflow 量：{sp.context_tokens - sp.context_limit:,} tokens",
                        ],
                        suggestion = (
                            "縮短 Context（壓縮申請資料 / 換 Summary Buffer），"
                            "或考慮換 Gemini 1.5 Pro 的 128K window"
                        ),
                    )

        # Rule 2：Tool Failure
        failed_tools = [
            sp for sp in spans
            if sp.operation == "tool_call" and sp.status == "error"
        ]
        if failed_tools:
            tool = failed_tools[0]
            return ErrorDiagnosis(
                trace_id   = trace_id,
                error_type = ErrorType.TOOL_FAILURE,
                confidence = 0.95,
                evidence   = [
                    f"工具 '{tool.tool_name}' 呼叫失敗",
                    f"錯誤訊息：{tool.tool_error or 'unknown'}",
                    f"Duration：{tool.duration_ms:.0f}ms",
                ],
                suggestion = (
                    f"檢查 '{tool.tool_name}' 的實作；"
                    "建議加 exponential backoff retry（max 3 次）；"
                    "考慮 circuit breaker 避免級聯失敗"
                ),
            )

        # Rule 3：Timeout
        timeout_spans = [
            sp for sp in spans if sp.status == "timeout"
        ]
        if timeout_spans:
            ts = timeout_spans[0]
            return ErrorDiagnosis(
                trace_id   = trace_id,
                error_type = ErrorType.TIMEOUT,
                confidence = 0.90,
                evidence   = [
                    f"operation='{ts.operation}' 發生 timeout",
                    f"TTFT={ts.ttft_ms:.0f}ms（超過正常 300ms 三倍以上）" if ts.ttft_ms else "TTFT 未記錄",
                ],
                suggestion = (
                    "短期：加 30 秒 request timeout + 自動 retry with Flash 模型作為 fallback；"
                    "長期：評估是否換 Gemini Flash 處理這類請求（P99 TTFT 較低）"
                ),
            )

        # Rule 4：Hallucination（output tokens 過少）
        llm_spans = [sp for sp in spans if sp.operation == "llm_call" and sp.output_tokens is not None]
        for sp in llm_spans:
            if sp.output_tokens < self.HALLUCINATION_OUTPUT_THRESHOLD and sp.status == "ok":
                return ErrorDiagnosis(
                    trace_id   = trace_id,
                    error_type = ErrorType.HALLUCINATION,
                    confidence = 0.75,
                    evidence   = [
                        f"LLM output_tokens={sp.output_tokens}（低於閾值 {self.HALLUCINATION_OUTPUT_THRESHOLD}）",
                        "結構化輸出可能被截斷，JSON 解析可能失敗",
                        f"model={sp.model}",
                    ],
                    suggestion = (
                        "加強 Output Format 約束（JSON schema / Pydantic validator）；"
                        "在 Prompt 加 Few-shot examples of complete output；"
                        "若持續發生考慮換 gemini-1.5-pro 替換 Flash"
                    ),
                )

        # Rule 5：Unknown error
        return ErrorDiagnosis(
            trace_id   = trace_id,
            error_type = ErrorType.UNKNOWN,
            confidence = 0.5,
            evidence   = ["偵測到錯誤 span，但無法匹配已知錯誤類型"],
            suggestion = "手動檢查此 trace 的完整 span 資料",
        )

    def classify_all(self, all_traces: dict[str, list[TraceSpan]]) -> list[ErrorDiagnosis]:
        diagnoses = []
        for trace_id, spans in all_traces.items():
            d = self.classify(trace_id, spans)
            if d:
                diagnoses.append(d)
        return diagnoses


classifier = ErrorClassifier()
diagnoses  = classifier.classify_all(all_traces)

# 統計各錯誤類型
type_counts: Counter = Counter(d.error_type for d in diagnoses)
total_failures = len(diagnoses)
total_traces   = len(all_traces)

print(f"錯誤自動分類報告（共 {total_traces} 筆，{total_failures} 筆失敗，失敗率 {total_failures/total_traces*100:.1f}%）")
print("=" * 65)
for etype, count in type_counts.most_common():
    bar = "█" * count
    print(f"  {etype.value:<25} {count:>3} 筆 ({count/total_traces*100:>4.1f}%)  {bar}")

print()
print("── 各類型詳細診斷範例 ─────────────────────────────────────")
seen_types = set()
for d in diagnoses:
    if d.error_type not in seen_types:
        seen_types.add(d.error_type)
        print(f"\n[{d.error_type.value}] trace={d.trace_id}  confidence={d.confidence:.0%}")
        for ev in d.evidence:
            print(f"  evidence: {ev}")
        print(f"  建議: {d.suggestion[:80]}..." if len(d.suggestion) > 80 else f"  建議: {d.suggestion}")

print()
print("為什麼自動分類比人工看 log 好？")
print(f"  人工分析 300 筆/日：假設每筆 3 分鐘 → {300 * 3 / 60:.0f} 小時/日")
print(f"  自動分類 300 筆：   < 100ms")
print(f"  工程師可以直接看分類報告，只需 debug 高優先度的 UNKNOWN 類型")

## 決策 5：Cloud Trace + Cloud Logging vs. 自建 Jaeger/Zipkin

**❌ 被拒絕的方案：自建 Jaeger 或 Zipkin**  
Jaeger 是優秀的開源分散式追蹤工具，但在金融機構的 GCP 環境下有幾個問題：
- **基礎設施維護**：需要部署 Jaeger collector、query、storage（Cassandra/Elasticsearch），  
  另外管理 HA、備份、升級——在 FinTech 的 compliance 環境這是額外的稽核負擔
- **整合成本**：與 Cloud Logging、Cloud IAM、VPC-SC 不整合，需要額外建 bridge
- **GCP 金融合規**：VPC Service Controls 在 Cloud Trace 原生支援，  
  自建 Jaeger 需要額外設計 perimeter 規則

**✅ 選擇的方案：Cloud Trace + Cloud Logging（GCP native）**

| 面向 | Cloud Trace + Logging | 自建 Jaeger |
|------|----------------------|-------------|
| 維護成本 | 零（GCP 管理） | 高（collector + storage） |
| GCP IAM 整合 | 原生 | 需要額外配置 |
| VPC-SC 支援 | 原生 | 需要 allow list |
| Cloud Monitoring 告警 | 原生整合 | 需要 push metrics |
| 同帳單 | 是 | 否（額外 infra 費用） |
| Trace 與 Log 相關聯 | 原生（trace_id 自動關聯） | 需要手動 log correlation |
| 定價 | $0.20/1M spans | Infra cost 不一定便宜 |

In [ ]:
import datetime

def span_to_cloud_trace_format(span: TraceSpan, project_id: str = "fintech-loan-prod") -> dict:
    """
    將內部 TraceSpan 轉換為 Cloud Trace API 格式。
    
    Cloud Trace 的核心欄位：
    - name：全域唯一的 span 識別路徑
    - spanId：16 字元 hex
    - parentSpanId：建立 span tree
    - displayName：UI 顯示的名稱
    - attributes：自訂屬性（LLM 相關指標放這裡）
    
    Cloud Logging 關聯：
    - 在 log entry 加上 trace 欄位，即可在 Cloud Logging UI 看到對應的 trace
    - 格式：projects/{project_id}/traces/{trace_id}
    """
    now_iso = datetime.datetime.utcnow().isoformat() + "Z"

    # 建構通用 attributes
    attributes: dict = {
        "/http/method":      {"stringValue": {"value": "POST"}},
        "agent.operation":   {"stringValue": {"value": span.operation}},
        "agent.status":      {"stringValue": {"value": span.status}},
    }

    # LLM-specific attributes
    if span.operation == "llm_call":
        if span.model:
            attributes["llm.model"] = {"stringValue": {"value": span.model}}
        if span.input_tokens:
            attributes["llm.input_tokens"] = {"intValue": str(span.input_tokens)}
        if span.output_tokens:
            attributes["llm.output_tokens"] = {"intValue": str(span.output_tokens)}
        if span.ttft_ms:
            attributes["llm.ttft_ms"] = {"intValue": str(int(span.ttft_ms))}
        if span.cost_usd:
            attributes["llm.cost_usd"] = {"stringValue": {"value": str(span.cost_usd)}}

    # Tool-specific attributes
    if span.operation == "tool_call":
        if span.tool_name:
            attributes["tool.name"]    = {"stringValue": {"value": span.tool_name}}
        attributes["tool.success"] = {"boolValue": bool(span.tool_success)}
        if span.tool_error:
            attributes["tool.error"] = {"stringValue": {"value": span.tool_error}}

    cloud_trace_span = {
        "name": f"projects/{project_id}/traces/{span.trace_id}/spans/{span.span_id}",
        "spanId": span.span_id.ljust(16, "0"),   # Cloud Trace 要求 16 hex chars
        "parentSpanId": (span.parent_span_id or "").ljust(16, "0"),
        "displayName": {
            "value": span.tool_name or span.model or span.operation,
            "truncatedByteCount": 0,
        },
        "startTime": now_iso,
        "endTime":   now_iso,
        "attributes": {"attributeMap": attributes},
        "status": {
            "code": 0 if span.status == "ok" else 2,  # 0=OK, 2=UNKNOWN error
            "message": span.status,
        },
    }
    return cloud_trace_span


def span_to_cloud_logging_entry(span: TraceSpan, project_id: str = "fintech-loan-prod") -> dict:
    """
    同一個 span 轉為 Cloud Logging 的結構化日誌格式。
    
    關鍵：trace 欄位讓 Cloud Logging 能關聯到 Cloud Trace。
    在 Cloud Logging UI 可以點「View in Trace」直接跳到對應 trace。
    """
    severity = "ERROR" if span.status in ("error", "timeout") else "INFO"

    structured_payload: dict = {
        "operation":  span.operation,
        "status":     span.status,
        "duration_ms": round(span.duration_ms, 2) if span.duration_ms else None,
    }
    if span.operation == "llm_call":
        structured_payload.update({
            "model":         span.model,
            "input_tokens":  span.input_tokens,
            "output_tokens": span.output_tokens,
            "ttft_ms":       round(span.ttft_ms, 1) if span.ttft_ms else None,
            "cost_usd":      span.cost_usd,
        })
    if span.operation == "tool_call":
        structured_payload.update({
            "tool_name":    span.tool_name,
            "tool_success": span.tool_success,
            "tool_error":   span.tool_error,
        })

    return {
        "severity":  severity,
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "jsonPayload": structured_payload,
        # ── 這兩個欄位讓 Logging 與 Trace 自動關聯 ──
        "trace":   f"projects/{project_id}/traces/{span.trace_id}",
        "spanId":  span.span_id.ljust(16, "0"),
        "traceSampled": True,
        # ── 資源標籤（方便 Cloud Logging 過濾）──
        "resource": {
            "type": "cloud_run_revision",
            "labels": {
                "project_id":    project_id,
                "service_name":  "loan-review-agent",
                "revision_name": "loan-review-agent-v2",
            },
        },
    }


# ── 展示一個失敗 trace 的完整 export 格式 ────────────────────────────────
sample_fail_tid = trace_ids_by_scenario["tool_failure"][0]
sample_spans    = collector.get_trace(sample_fail_tid)

print(f"Cloud Trace Export 格式（trace_id={sample_fail_tid}）：")
print("─" * 60)
for sp in sample_spans[:2]:  # 只展示前兩個 span
    ct_span = span_to_cloud_trace_format(sp)
    print(json.dumps(ct_span, indent=2, ensure_ascii=False)[:600])
    print("...(截斷)")
    print()

print("\nCloud Logging 關聯格式：")
print("─" * 60)
tool_fail_spans = [sp for sp in sample_spans if sp.operation == "tool_call"]
if tool_fail_spans:
    log_entry = span_to_cloud_logging_entry(tool_fail_spans[0])
    print(json.dumps(log_entry, indent=2, ensure_ascii=False))

print()
print("Cloud Trace + Cloud Logging 的核心優勢：")
print("  1. trace 欄位讓 Log entry 直接連到 Trace waterfall")
print("  2. VPC-SC 原生支援，資料不出 GCP perimeter")
print("  3. Cloud Monitoring alerting policy 可直接讀 Trace 的 error rate")
print("  4. 同一帳單，不需要管理額外的 infra（Cassandra/Elasticsearch for Jaeger）")

## 完整 Debug 流程：從 15% 失敗到根因定位

In [ ]:
def full_debug_report(
    all_traces: dict[str, list[TraceSpan]],
    diagnoses: list[ErrorDiagnosis],
) -> None:
    """
    模擬 SRE 收到告警後的完整 debug 流程：
    1. 看 Error Taxonomy 摘要 → 了解失敗分布
    2. 看 LLM-native metrics → 定位效能瓶頸
    3. 看具體的 fail trace → 確認根因
    4. 輸出 action items
    """
    total = len(all_traces)
    errors = len(diagnoses)

    print("╔══════════════════════════════════════════════════════════╗")
    print("║  貸款審查 Agent — 生產 Debug 報告                       ║")
    print("║  觸發：失敗率 Alert（> 10% threshold）                  ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print()

    # Step 1：失敗率總覽
    print("【Step 1】失敗率分析")
    print(f"  總請求：{total}  失敗：{errors}  失敗率：{errors/total*100:.1f}%")
    print()

    # Step 2：Error Taxonomy
    print("【Step 2】自動錯誤分類（傳統 log 看不到的）")
    type_counts = Counter(d.error_type for d in diagnoses)
    for etype, count in type_counts.most_common():
        pct = count / total * 100
        print(f"  {etype.value:<25} {count:>3} 筆  {pct:>5.1f}%  {'▓' * count}")
    print()

    # Step 3：LLM-native 指標
    m = MetricsCollector(all_traces)
    print("【Step 3】LLM-native 效能指標")
    tool_rpt = m.tool_latency_report()
    for tool, stats in tool_rpt.items():
        flag = "⚠️ " if float(stats['error_rate'].replace('%', '')) > 5 else "  "
        print(f"  {flag}{tool:<28} P95={stats['p95_ms']:>7}ms  失敗率={stats['error_rate']}")
    ctx_rpt = m.context_utilization_report()
    overflow_rate = float(ctx_rpt['overflow_rate'].replace('%',''))
    flag = "⚠️ " if overflow_rate > 0 else "  "
    print(f"  {flag}Context Overflow              發生次數={ctx_rpt['overflow_count']}  "
          f"overflow_rate={ctx_rpt['overflow_rate']}")
    print()

    # Step 4：Action Items
    print("【Step 4】根因 & 行動建議（優先順序）")
    print()

    action_items = []
    for etype, count in type_counts.most_common():
        # 取第一個該類型的診斷取得建議
        example = next(d for d in diagnoses if d.error_type == etype)
        action_items.append((count, etype, example.suggestion))

    for priority, (count, etype, suggestion) in enumerate(action_items, 1):
        print(f"  #{priority} [{etype.value}]  影響 {count} 筆（{count/total*100:.1f}%）")
        # 分行顯示建議
        words = suggestion.split("；")
        for w in words:
            if w.strip():
                print(f"      → {w.strip()}")
        print()

    cost = m.cost_report()
    if cost:
        print(f"  月 LLM 成本：${cost['monthly_cost_usd']:,.2f}  "
              f"（修復失敗後可節省 ~${cost['monthly_cost_usd'] * errors/total * 0.3:.2f}/月）")


full_debug_report(all_traces, diagnoses)

## FDE 面試白板語言

以下是這個場景在 Google FDE 技術面試中最常被問到的問題，以及白板答題框架。

In [ ]:
print("""
FDE 面試：Agent 可觀測性的核心問題
═══════════════════════════════════════════════════════════

Q1: 生產環境 Agent 出現 15% 失敗率，你怎麼 debug？

A: 我的 debug 流程分三層：

   第一層：用 Error Taxonomy 摘要快速分類。
   先看失敗是集中在哪種類型——
   TOOL_FAILURE 代表外部 API 問題，
   CONTEXT_OVERFLOW 代表輸入設計問題，
   HALLUCINATION 代表 Prompt 問題。
   這一步告訴我「去哪裡找」，不需要讀 300 條 raw log。

   第二層：用 LLM-native metrics 確認瓶頸。
   看 TTFT P95、tokens/sec、tool P95 latency。
   比如如果 query_jcic P95 是 5200ms 而正常是 400ms，
   就可以確認是聯徵 API 的問題，而不是 LLM 問題。

   第三層：用 Tail-sampled trace 看具體案例。
   因為失敗 trace 100% 採樣，直接在 Cloud Trace 過濾
   error=true 就能看到完整的 span waterfall，
   定位到具體是第幾次 tool call 或 LLM call 失敗。

═══════════════════════════════════════════════════════════

Q2: 為什麼要用 LLM-native metrics 而不是傳統 APM？

A: 傳統 APM 的 HTTP latency 對 Agent 的指導價值很有限。
   一個 Agent 請求的 4 秒延遲，可能是：
   (a) LLM TTFT 慢（模型推理問題）→ 換 Flash 或 streaming
   (b) Tool DB 查詢慢（資料庫問題）→ 加 index 或 cache
   (c) Context 太大（設計問題）→ 縮 context
   這三個原因的修復方向完全不同，HTTP latency 無法區分。

   LLM-native metrics 給出三個 HTTP 沒有的維度：
   1. TTFT 分離感知延遲和吞吐延遲
   2. tokens/sec 偵測模型端資源競爭
   3. cost/request 把 LLM 費用拉進 SLO，
      讓你知道某個場景是否值得用 Pro 模型

═══════════════════════════════════════════════════════════

Q3: Tail-based sampling 的 sampling rate 怎麼決定？

A: 這是個 cost vs. debuggability 的 trade-off，
   我的決策框架是：

   失敗 / 慢請求（P95 以上）：永遠 100%。
   這些是你最需要的，存儲成本遠低於一個未知 bug 的損失。

   正常成功請求：5% baseline 採樣。
   原因：你需要一個 baseline 來偵測 performance drift，
   比如本週 P50 TTFT 比上週高 20%，這需要有成功 trace 才能比較。
   5% 是我們算過的：每日 100 筆正常 trace 足夠 drift 偵測，
   但不超出 Cloud Trace 的 free tier（每月 2.5M spans）。

   如果問「為什麼不用 Head-based？」
   因為 Head-based 在請求進入時決定，不知道結果。
   10% Head-based = 有 90% 機率丟掉失敗 trace。
   Tail-based 在結果知道後才決定，失敗的 100% 保留。

═══════════════════════════════════════════════════════════

Q4: Cloud Trace vs. 自建 Jaeger，你怎麼選？

A: 對金融機構在 GCP 上的場景，我選 Cloud Trace，
   有三個決定性因素：

   1. VPC Service Controls：
      金融機構需要 VPC-SC 確保資料不出 perimeter，
      Cloud Trace 原生支援，自建 Jaeger 需要在 allow list
      加規則，增加 surface area 和 compliance 風險。

   2. Log-Trace 關聯：
      Cloud Logging 的 log entry 加上 trace 欄位後，
      直接在 UI 可以 click 到 Trace waterfall，
      Jaeger 需要另外 build 這個 correlation。

   3. 零 ops 負擔：
      Jaeger 需要維護 collector + Cassandra/Elasticsearch，
      在金融機構的 change management 流程下，
      每次 Jaeger 升級都是一個 RFC，不值得。

   什麼時候選 Jaeger？
   多雲環境（跨 GCP + AWS）、或組織已有 Jaeger 投入，
   遷移成本高於維護成本時。
═══════════════════════════════════════════════════════════
""")

## 架構決策速查表

| 決策 | 選擇 | 核心理由 | 反例（什麼時候選另一個） |
|------|------|----------|-------------------------|
| 追蹤粒度 | Granular Span | Request-level 無法定位是哪一步失敗 | 極簡 script，不需要 debug |
| 採樣策略 | Tail-based（失敗100%/成功5%） | Head-based 會丟失 90% 的失敗案例 | 所有請求都是高優先度（金融審計要求100%） |
| 效能指標 | LLM-native（TTFT/cost/tokens） | HTTP latency 無法指出是 LLM 慢還是 tool 慢 | 純 API proxy，沒有 LLM 組件 |
| 錯誤分類 | 自動 Taxonomy | 300筆/日的手動分析需要 15 小時 | 請求量極小（< 10/日），值得人工分析 |
| 追蹤後端 | Cloud Trace + Cloud Logging | VPC-SC原生、Log-Trace關聯、零 ops | 多雲環境或已有 Jaeger 大量投入 |